In [ ]:
import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '6'

from dotenv import load_dotenv
from collections.abc import Sequence
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

import flax.jax_utils as flax_utils
import flax.linen as nn
import grain.python as grain
import jax
import numpy as np
from absl import logging
from connectomics.jax import checkpoint, training
from etils import epath
from orbax import checkpoint as ocp


from zapbench.ts_forecasting import heads, input_pipeline, train
from zapbench.ts_forecasting.configs import infer, mean, linear, timemix, tsmixer, tide
from zapbench.ts_forecasting.infer_idx import _filter_infer_indices
import zapbench.models.util as model_util

import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])

load_dotenv()
PATH = os.getenv("ROOT_PATH")
LOG_PATH = os.getenv("LOG_PATH")

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  # Set all spines invisible
  for spine in ax.spines.values():
    spine.set_visible(False)
  # Hide all ticks and tick labels
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
def _get_checkpoint_step(
    checkpoint_manager: ocp.CheckpointManager,
    selection_strategy: str,
) -> int | None:
  """Returns the checkpoint step to use given a selection strategy.

  Args:
    checkpoint_manager: Checkpoint manager.
    selection_strategy: Checkpoint selection strategy, can be 'early_stopping',
      'best_val_loss', or 'latest'.

  Returns:
    Checkpoint step.
  """
  if selection_strategy == 'early_stopping':
    checkpointed_state = dict(
        early_stop=None,
    )
    checkpointed_state = checkpoint.restore_checkpoint(
        checkpoint_manager,
        state=checkpointed_state,
        step=checkpoint_manager.latest_step(),
    )
    return checkpointed_state['early_stop']['best_step']
  elif selection_strategy == 'best_val_loss':
    checkpointed_state = dict(
        track_best_val_loss_step=None,
    )
    checkpointed_state = checkpoint.restore_checkpoint(
        checkpoint_manager,
        state=checkpointed_state,
        step=checkpoint_manager.latest_step(),
    )
    return checkpointed_state['track_best_val_loss_step']['best_step']
  elif selection_strategy == 'latest':
    return checkpoint_manager.latest_step()
  else:
    raise ValueError(f'Unknown checkpoint selection: {selection_strategy}')


def infer_single_step(
    model: nn.Module,
    head: heads.Head,
    train_state: train.TrainState,
    data_source: grain.RandomAccessDataSource,
    idx: int,
    infer_key: jax.Array,  # pylint: disable=unused-argument
    covariates: Sequence[str] = (),
    covariates_static: jax.Array | None = None,
    with_carry: bool = False,
) -> tuple[jax.Array, jax.Array]:
  """Runs independent inference on each index in the test set.

  Returns:
    prediction: prediction array
    target: target array
  """
  carry = None

  batch = data_source[idx]
  if 'covariates_static' in covariates:
    batch['covariates_static'] = covariates_static

  out = train.pred_step(
      model,
      train_state,
      batch,
      covariates,
      initial_carry=carry,
      return_carry=with_carry,
  )

  if not with_carry:
    dist = head.get_distribution(out)
  else:
    carry, dist = out[0], head.get_distribution(out[1])

  prediction = dist.mode()
  target = batch['timeseries_output']

  return prediction, target

In [ ]:
exp_workdir = f'/{LOG_PATH}/timemix/subject_05/'
exp_config = model_util.load_config(os.path.join(exp_workdir, 'config.json'))

model = model_util.model_from_config(exp_config)

covariates_static = input_pipeline.get_static_covariates(exp_config)

checkpoint_manager = checkpoint.get_checkpoint_manager(
    exp_workdir,
    item_names=(
        'early_stop',
        'train_state',
        'track_best_val_loss_step',
    ),
)

step = _get_checkpoint_step(checkpoint_manager, 'best_val_loss')

checkpointed_state = dict(
    train_state=None,
)
checkpointed_state = checkpoint.restore_checkpoint(
    checkpoint_manager,
    state=checkpointed_state,
    step=step,
)
train_state = checkpointed_state['train_state']
train_state = train.TrainState(
    **train_state
)
train_state = flax_utils.replicate(train_state)

In [ ]:
# inference
infer_config = infer.get_config()
config = timemix.get_config('dataset_name=subject_06,timesteps_input=32')
config.update(infer_config)

head = heads.create_head(config)
rng = training.get_rng(config.seed)
rng, infer_rng = jax.random.split(rng)
infer_source = input_pipeline.create_inference_source_with_transforms(config)
infer_key = jax.random.fold_in(key=infer_rng, data=step)

Infer selected neurons at all timepoints

In [ ]:
context = config.timesteps_input_infer + config.timesteps_output_infer
prediction_list, target_list = [], []
full_prediction, full_target, transition_idx = [], [], []
counter = 0
for infer_idx_set in config.infer_idx_sets:
  name, idx_list = (infer_idx_set[k] for k in ('name', 'idx_list'))
  idx_list = _filter_infer_indices(idx_list, context)
  infer_metrics = None
  train_state = train.merge_batch_stats(train_state)
  for i, idx in enumerate(idx_list):
    prediction, target = infer_single_step(
        model,
        head,
        flax_utils.unreplicate(train_state),
        infer_source,
        idx,
        infer_key=infer_key,
        covariates=tuple(config.covariates),
        covariates_static=covariates_static,
        with_carry=config.infer_with_carry,
    )
    prediction_list.append(prediction[0, [0, 16], ::5000])
    target_list.append(target[0, [0, 16], ::5000])

    if counter%36==0:
      nans = np.full(infer_source[idx]['timeseries_input'][0, :4, ::5000].shape, np.nan)
      full_prediction.append(np.concatenate([nans, prediction[0, :32, ::5000]], 0))
      full_target.append(np.concatenate([infer_source[idx]['timeseries_input'][0, :4, ::5000], infer_source[idx]['timeseries_output'][0, :32, ::5000]], 0))
      # transition_idx.append(i)
    counter += 1
  full_prediction.append(np.full(prediction[0, :32, ::5000].shape, np.nan))
  full_target.append(np.full(infer_source[idx]['timeseries_output'][0, :32, ::5000].shape, np.nan))

full_prediction = np.concatenate(full_prediction)
full_target = np.concatenate(full_target)

In [ ]:
prediction_list = np.stack(prediction_list)
target_list = np.stack(target_list)
prediction_list.shape
target_list.shape

In [ ]:
fig, ax = plt.subplots(prediction_list.shape[-1], 1, figsize=(10, 15), dpi=400)
for i_neuron, a in zip(range(prediction_list.shape[-1]), ax):
  a.plot(prediction_list[:, 0, i_neuron], color='red', linewidth=1.5)
  a.plot(target_list[:, 0, i_neuron], color='k', alpha=0.8)
  format_ax(a)

In [ ]:
fig, ax = plt.subplots(prediction_list.shape[-1], 1, figsize=(10, 15), dpi=400)
for i_neuron, a in zip(range(prediction_list.shape[-1]), ax):
  a.plot(prediction_list[1:, 0, i_neuron]-target_list[:-1, 0, i_neuron], color='k')
  format_ax(a)

In [ ]:
fig, ax = plt.subplots(prediction_list.shape[-1], 1, figsize=(10, 15), dpi=400)
for i_neuron, a in zip(range(prediction_list.shape[-1]), ax):
  a.plot(prediction_list[:, -1, i_neuron], color='red', linewidth=1.5)
  a.plot(target_list[:, -1, i_neuron], color='k', alpha=0.8)
  format_ax(a)

In [ ]:
fig, ax = plt.subplots(full_prediction.shape[-1], 1, figsize=(10, 15), dpi=400)
for i_neuron, a in zip(range(full_prediction.shape[-1]), ax):
  a.plot(full_prediction[:, i_neuron], color='red', linewidth=1.5)
  a.plot(full_target[:, i_neuron], color='k', alpha=0.8)
  # for idx in transition_idx:
  #   a.axvline(idx, color='purple', linestyle='-', linewidth=1)
  format_ax(a)

Predictions for selected indices in optic processing regions

In [ ]:
selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128]

In [ ]:
context = config.timesteps_input_infer + config.timesteps_output_infer
prediction_list, target_list = [], []
full_prediction, full_target, transition_idx = [], [], []
counter = 0

infer_metrics = None
train_state = train.merge_batch_stats(train_state)
for idx in range(500):
  prediction, target = infer_single_step(
      model,
      head,
      flax_utils.unreplicate(train_state),
      infer_source,
      idx,
      infer_key=infer_key,
      covariates=tuple(config.covariates),
      covariates_static=covariates_static,
      with_carry=config.infer_with_carry,
  )
  prediction_list.append(prediction[0, 8, selected_ix])
  target_list.append(target[0, 8, selected_ix])

In [ ]:
prediction_list = np.stack(prediction_list)
target_list = np.stack(target_list)

In [ ]:
n_ix = 50
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i in range(n_ix):
    ax = axs[i]
    ax.plot(target_list[:, i], 'k')
    ax.plot(prediction_list[:, i], 'r')
    format_ax(ax)

plt.tight_layout()

MAE across all neurons

In [ ]:
context = config.timesteps_input_infer + config.timesteps_output_infer
all_predictions_0, all_targets_0 = [], []
all_predictions_32, all_targets_32 = [], []
cumulative_abs_error = []
target_varibility = []
# for infer_idx_set in config.infer_idx_sets:
# name, idx_list = (infer_idx_set[k] for k in ('name', 'idx_list'))
idx_list = config.infer_idx_sets[-1]['idx_list']
idx_list = _filter_infer_indices(idx_list, context)
infer_metrics = None
train_state = train.merge_batch_stats(train_state)
for i, idx in enumerate(idx_list):
  print(idx)
  prediction, target = infer_single_step(
      model,
      head,
      flax_utils.unreplicate(train_state),
      infer_source,
      idx,
      infer_key=infer_key,
      covariates=tuple(config.covariates),
      covariates_static=covariates_static,
      with_carry=config.infer_with_carry,
  )
  all_predictions_0.append(prediction[0,0])
  all_targets_0.append(target[0,0])
  all_predictions_32.append(prediction[0,-1])
  all_targets_32.append(target[0,-1])
  cumulative_abs_error.append(np.abs(prediction[0,0]-target[0,0]))
  target_varibility.append(target[0,0])

In [ ]:
all_predictions_0 = np.array(all_predictions_0)
all_targets_0 = np.array(all_targets_0)
all_predictions_32 = np.array(all_predictions_32)
all_targets_32 = np.array(all_targets_32)
cumulative_abs_error = np.array(cumulative_abs_error)
target_varibility = np.var(target_varibility, axis=0)

np.abs(all_predictions_0 - all_targets_0).mean(), np.abs(all_predictions_32 - all_targets_32).mean()

Mean

In [ ]:
np.abs(all_predictions_0 - all_targets_0).mean(), np.abs(all_predictions_32 - all_targets_32).mean()

Linear

In [ ]:
np.abs(all_predictions_0 - all_targets_0).mean(), np.abs(all_predictions_32 - all_targets_32).mean()

Timemix

In [ ]:
np.abs(all_predictions_0 - all_targets_0).mean(), np.abs(all_predictions_32 - all_targets_32).mean()

In [ ]:
02: (np.float32(0.019320523), np.float32(0.031962115))
pretrain: (np.float32(0.019420652), np.float32(0.031274892))
zapbench: (np.float32(0.019774755), np.float32(0.032930132))
after 500 (np.float32(0.021646533), np.float32(0.034482602))


Tsmixer

In [ ]:
np.abs(all_predictions_0 - all_targets_0).mean(), np.abs(all_predictions_32 - all_targets_32).mean()

In [ ]:
abs_diff = np.abs(prediction - target)[0, :, :]
plt.figure(figsize=(5, 5), dpi=200)
plt.imshow(abs_diff.T, aspect='auto', cmap='magma', origin='lower', vmin=0)
plt.colorbar()
plt.axis('off')
plt.show()

Show error on anatomy

In [ ]:
import scipy, h5py

reference_anat = scipy.io.loadmat(f'{PATH}/Additional_mat_files/ReferenceBrain.mat')

subject_id = 15

h5_path = f"{PATH}/subject_{subject_id:02d}"
print(h5_path)
h5 = h5py.File(f"{h5_path}/TimeSeries.h5", "r")
abs_ix = h5['absIX']
abs_ix = (abs_ix[0] - 1).astype(int)

mat_path = f"{PATH}/subject_{subject_id:02d}/data_full.mat"
data_struct = scipy.io.loadmat(mat_path)['data'][0, 0]

all_cell_coordinates = data_struct[8]
coordinates = all_cell_coordinates[abs_ix]

plt.figure(figsize=(10, 10))
plt.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
abs_deviation = np.abs(prediction[0]-target[0])
abs_deviation.shape

Error t=1 and t=32

In [ ]:
for i in range(0, 100, 1):
  fig, axs = plt.subplots(2, 1, figsize=(20, 10))
  axs[0].imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
  sc0 = axs[0].scatter(
      coordinates[:, 0], coordinates[:, 1],
      c=np.log(np.abs(all_targets_0[i]-all_predictions_0[i])),
      cmap='coolwarm',
      s=0.1,
      vmin=np.log(abs_deviation).min(),
      vmax=np.log(abs_deviation).max()
  )
  axs[0].axis('off')

  axs[1].imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
  sc1 = axs[1].scatter(
      coordinates[:, 0], coordinates[:, 1],
      c=np.log(np.abs(all_targets_32[i]-all_predictions_32[i])),
      cmap='coolwarm',
      s=0.1,
      vmin=np.log(abs_deviation).min(),
      vmax=np.log(abs_deviation).max()
  )
  axs[1].axis('off')

  plt.show()
  clear_output(wait=True)
  plt.close()

Cumulative error t=0

In [ ]:
cumulative_abs_error.mean(0)
fig, axs = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
axs.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
sc0 = axs.scatter(
    coordinates[:, 0], coordinates[:, 1],
    c=np.log(cumulative_abs_error.mean(0)),
    cmap='Reds',
    s=0.2,
    alpha=0.5,
)
plt.colorbar(sc0, ax=axs, fraction=0.025, pad=0.04, aspect=30, shrink=1.0).ax.tick_params(labelsize=18)
plt.tight_layout()
axs.axis('off');

Compare with average variability per region (ie, is the model error meaningless in the sense that it is just a reflection of the variability of the data?) 

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 5), dpi=300)
axs.imshow(np.sum(reference_anat['anat_stack_norm'], axis=-1).T, cmap='gray')
sc0 = axs.scatter(
    coordinates[:, 0], coordinates[:, 1],
    c=np.log(target_varibility),
    cmap='coolwarm',
    s=0.2,
)
plt.colorbar(sc0, ax=axs, fraction=0.025, pad=0.04, aspect=30, shrink=1.0).ax.tick_params(labelsize=18)
plt.tight_layout()
axs.axis('off');

In [ ]:
np.corrcoef(cumulative_abs_error.mean(0), target_varibility)

In [ ]:
import pyvista as pv
import matplotlib.pyplot as plt
import numpy as np

x = reference_anat['anat_stack_norm']
grid = pv.wrap(x)
grid.spacing = [1.0, 1.0, 2.0]

opacity = [
    0.0,
    0.1,
    0.3,
    0.5,
    0.7,
]

points = coordinates * np.array(grid.spacing)
colors = np.log(target_varibility)

point_cloud = pv.PolyData(points)

# Use an off-screen plotter for static image
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='hot',
    point_size=3,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.add_volume(
    grid,
    cmap="gray",
    opacity=opacity,
    shade=True,
    show_scalar_bar=False,
)

plotter.camera_position = 'yz'
plotter.camera.azimuth = -150
plotter.camera.elevation = 30
plotter.camera.zoom(1.5)
plotter.show()

In [ ]:
import pyvista as pv
import matplotlib.pyplot as plt
import numpy as np

x = reference_anat['anat_stack_norm']
grid = pv.wrap(x)
grid.spacing = [1.0, 1.0, 2.0]

opacity = [
    0.0,
    0.1,
    0.3,
    0.5,
    0.7,
]
# opacity = [
#     0.0,
#     0.0,
#     0.05,
#     0.05,
#     0.1,
# ]

points = coordinates * np.array(grid.spacing)
colors = np.log(target_varibility)

point_cloud = pv.PolyData(points)

plotter = pv.Plotter(off_screen=True, window_size=(3000, 3000))
plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='hot',
    point_size=6,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.add_volume(
    grid,
    cmap="gray",
    opacity=opacity,
    shade=True,
    show_scalar_bar=False,
)

plotter.camera_position = 'yz'
plotter.camera.azimuth = -150
plotter.camera.elevation = 30
plotter.camera.zoom(1.5)

# Save the image to file in higher quality (larger resolution and higher DPI)
img_path = "variability.png"
plotter.screenshot(img_path, window_size=(1300, 800), scale=3.0)
plotter.close()
print(f"Saved high-quality image to {img_path}")

In [ ]:
# Use an off-screen plotter for static image
plotter = pv.Plotter(off_screen=True, window_size=(3000, 3000))
plotter.add_mesh(
    point_cloud,
    scalars=colors,
    cmap='magma',
    point_size=6,
    render_points_as_spheres=True,
    show_scalar_bar=False
)

plotter.add_volume(
    grid,
    cmap="gray",
    opacity=opacity,
    shade=True,
    show_scalar_bar=False,
)

plotter.camera_position = 'xy'
plotter.camera.azimuth = 0
plotter.camera.elevation = 0
plotter.camera.zoom(1.8)

# Save the image to file in higher quality (larger resolution and higher DPI)
img_path = "harmonised_pointcloud.png"
plotter.screenshot(img_path, window_size=(1300, 800), scale=3.0)
plotter.close()
print(f"Saved high-quality image to {img_path}")

In [ ]:
import glob
from zapbench.ts_forecasting import util

path_to_inference = glob.glob("/mnt/storage/misc/zapbench/inference/n_steps_4/mean/**/", recursive=True)[-1]
df = util.get_per_step_metrics_from_directory(path_to_inference,metric='MAE')
df.sort_values('condition')

In [ ]:
for condition in df["condition"].unique():
  print(df[df["condition"] == condition]['MAE'].mean())

In [ ]:
import pandas as pd
from connectomics.common import ts_utils
from zapbench import constants

df = pd.DataFrame(
    ts_utils.load_json(f'gs://zapbench-release/dataframes/20250131/combined.json'))
df.head()

In [ ]:
df[(df['condition']=='gain') & (df['method']=='mean') & (df['context']==4)]

In [ ]:
idx_list = config.infer_idx_sets[-1]['idx_list']
print(idx_list)

In [ ]:
idx_list = _filter_infer_indices(idx_list, context)

In [ ]:
from zapbench import data_utils

data_utils.adjust_condition_bounds_for_split(
    'test_holdout',
    3078,
    3735,
    4)

In [ ]:
def get_infer_sets(
    num_timesteps_context: int,
    dataset_name: str = constants.DEFAULT_DATASET,
) -> Sequence[dict[str, int | str]]:
  """Get infer sets config with dataset-aware conditions."""
  dataset_config = constants.get_dataset_config(dataset_name)
  conditions_train = dataset_config['conditions_train']
  conditions_holdout = dataset_config['conditions_holdout']

  sets = []
  for condition, split in [(t, 'test') for t in conditions_train] + [
      (t, 'test_holdout') for t in conditions_holdout
  ]:
    inclusive_min, exclusive_max = data_utils.adjust_condition_bounds_for_split(
        split,
        *data_utils.get_condition_bounds(condition, dataset_name=dataset_name),
        num_timesteps_context=num_timesteps_context,
    )
    sets.append({
        'name': f'{split}_condition_{condition}',
        'start_idx': inclusive_min,
        'num_windows': data_utils.get_num_windows(
            inclusive_min, exclusive_max, num_timesteps_context
        ),
    })
  return sets

In [ ]:
infer_sets = get_infer_sets(
  num_timesteps_context=config.timesteps_input_infer
  + config.num_warmup_infer_steps,
  dataset_name=constants.DEFAULT_DATASET,
)

In [ ]:
start_idx = infer_sets[-1]['start_idx']
start_idx

In [ ]:
num_steps=(
    config.num_warmup_infer_steps +
    config.prediction_window_length //
    config.timesteps_output_infer)
num_steps, config.prediction_window_length, config.timesteps_output_infer

Up to this point, everything agrees perfectly... what is going on in infer?

In [ ]:
         predictions, targets = infer(
              model,
              head,
              flax_utils.unreplicate(train_state),
              infer_source,
              start_idx=start_idx + window,
              num_steps=(
                  config.num_warmup_infer_steps +
                  config.prediction_window_length //
                  config.timesteps_output_infer),
              prediction_window_length=config.prediction_window_length,
              infer_key=infer_key,

In [ ]:

start_idx = infer_sets[-1]['start_idx']
num_windows = infer_sets[-1]['num_windows']
for window in range(num_windows):
  real_start_idx = start_idx + window
  t_axis = -2  # Assume shape ...xTxF
  t_in = infer_source[0]['timeseries_input'].shape[t_axis]
  t_out = infer_source[0]['timeseries_output'].shape[t_axis]
  end_idx = min(len(infer_source), real_start_idx + num_steps * t_out)
  series_input_override, carry = None, None
  predictions, targets = [], []
  all_indices = []
  for i, idx in enumerate(range(start_idx, end_idx, t_out)):
    all_indices.append(idx)

In [ ]:
print(all_indices)

In [ ]:
print(idx_list[::32])